# 05 — Theo dõi từng bước tách QRS và T loops trên dữ liệu thật

Notebook này giải thích **vì sao các mốc được chọn và cách các mẫu XYZ tạo thành loop**, bằng một recording xuyên suốt. Mặc định là HC `patient104/s0306lre`; mọi số trên hình được tính từ recording này, không vẽ một nhịp giả.

Ý tưởng xuyên suốt:

**Một recording có nhiều nhịp → chọn các nhịp qua QC dữ liệu → một median beat → chọn hai đoạn thời gian → nối điểm XYZ trong mỗi đoạn → hai loop.**

Điện thế theo thời gian, magnitude, vận tốc và loop là những cách biểu diễn khác nhau của dữ liệu liên quan. Biên phân đoạn là ước lượng theo thuật toán notebook 04, chưa phải annotation chuẩn. Đặc biệt, T-on/end theo tiếp tuyến là một quy ước đo; không nhất thiết bao hết phần đuôi nhỏ của sóng.

Đọc lần lượt từ Bước 1. Mỗi bước có **đầu vào**, **thao tác**, **đầu ra** và hướng dẫn đọc biểu đồ. Các hình và bảng đã được lưu sẵn. Muốn đổi record, sửa `RECORD_ID`, chọn kernel Python (ipcv), rồi Run All.

## Các hàm nền dùng chung với notebook 04

Ba module bên cạnh notebook cung cấp tiền xử lý có điều kiện, pipeline và kiểm chứng. Mở notebook từ thư mục project, chọn Python (ipcv); restart kernel trước Run All để tránh dùng module đã cache. Thu gọn ô import và đọc từ Bước 1.

In [ ]:
# %pip install numpy scipy pandas matplotlib scikit-learn wfdb
from pathlib import Path
from dataclasses import asdict, replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, savgol_filter, butter, sosfiltfilt
from scipy.ndimage import uniform_filter1d, percentile_filter, binary_closing
from sklearn.decomposition import PCA
from vcg_preprocessing import (Config, legacy_config, design_morphology_fir,
    analyze_powerline_noise, decide_notch, assess_input_quality,
    filter_frequency_response, welch_xyz)
from vcg_pipeline import (FEATURE_NAMES, preprocess_xyz, ms_samples, odd_window, robust_sigma, load_xyz, detect_qrs_anchors, spatial_speed, flattest_pq, representative_beat, quiet_boundary, delineate_loops, apply_manual_boundaries, cut_loop, project_pca, extract_features, process_record, qc_row, plot_detection, plot_delineation, plot_loops)
from vcg_preprocessing_evaluation import (compare_preprocessing_configs,
    powerline_diagnostic_table, plot_filter_responses, plot_preprocessing_diagnostics,
    plot_median_config_comparison, plot_3d_config_comparison)
CFG=Config()


In [ ]:
from IPython.display import display
from matplotlib.patches import Rectangle

DATA_ROOT = Path('content/ptbdb/physionet.org/files/ptbdb/1.0.0')
RECORD_ID = 'patient104/s0306lre'
CFG = Config(seconds=30.0)
QRS_COLOR, T_COLOR = '#059669', '#ea580c'
XYZ_COLORS = ['#2563eb', '#7c3aed', '#64748b']
plt.rcParams.update({'font.family':'DejaVu Sans', 'figure.dpi':105, 'font.size':10})

def finish(fig):
    plt.show()
    plt.close(fig)

def draw_xyz(ax, time, xyz, label_prefix=''):
    for j,name in enumerate('XYZ'):
        ax.plot(time,xyz[:,j],color=XYZ_COLORS[j],lw=1.3,label=label_prefix+name)
    ax.set_ylabel('Điện thế (mV)')
    ax.grid(alpha=.2)

def shade(ax, time, bounds):
    for a,z,color,label in [('qrs_on','qrs_off',QRS_COLOR,'QRS candidate'),('t_on','t_off',T_COLOR,'T candidate')]:
        if bounds[a] is not None and bounds[z] is not None:
            ax.axvspan(time[bounds[a]],time[bounds[z]],color=color,alpha=.16,label=label)
            ax.axvline(time[bounds[a]],color=color,ls=':',lw=1)
            ax.axvline(time[bounds[z]],color=color,ls=':',lw=1)

if not DATA_ROOT.is_dir():
    raise FileNotFoundError('Mở notebook từ thư mục project hoặc sửa DATA_ROOT.')

## Bước 1 — Một hàng dữ liệu XYZ có nghĩa gì

**Đầu vào:** record PTB gồm header và tín hiệu. Chỉ ba kênh Frank `vx/vy/vz` được đọc.

**Thao tác:** đọc ba cột điện thế đồng thời. Mỗi hàng ứng với một thời điểm; ba giá trị trên hàng đó tạo thành một điểm $(X,Y,Z)$ trong không gian điện thế.

**Đầu ra:** `raw_xyz` có shape `(số mẫu, 3)` và tần số lấy mẫu `fs`. Ở 1000 Hz, hai hàng liên tiếp cách nhau 1 ms.

Ví dụ, hàng có X=0.2, Y=0.3, Z=0.1 mV là điểm (0.2, 0.3, 0.1). Điểm này không phải vị trí giải phẫu trong tim; các tọa độ đều là điện thế. Bảng dưới dùng số thật từ recording.

In [ ]:
raw = load_xyz(DATA_ROOT / RECORD_ID, CFG.seconds)
raw_xyz, fs = raw['xyz'], raw['fs']
print('Record:', RECORD_ID, '| diagnosis:', raw['diagnosis'])
print('Channels đã đọc:', raw['channels'])
print('Shape:', raw_xyz.shape, '| fs:', fs, 'Hz | Δt:', 1000/fs, 'ms')
rows = np.arange(int(2*fs), int(2*fs)+5)
display(pd.DataFrame({'chỉ số mẫu':rows,'thời gian recording (s)':rows/fs,
                      'X (mV)':raw_xyz[rows,0],'Y (mV)':raw_xyz[rows,1],'Z (mV)':raw_xyz[rows,2]}))

### Bước 1A — Kiểm tra chất lượng XYZ trước lọc

**Đầu vào:** XYZ gốc. **Thao tác:** kiểm tra hữu hạn, đoạn cả ba trục gần như hằng, đột biến đơn lẻ, vùng gần rail ADC danh nghĩa PTB, PSD tần số thấp/cao và thay đổi baseline giữa các khối.

**Đầu ra:** dictionary QC/mask để xem và dùng kiểm tra beat. Đỉnh QRS dốc hoặc hình thái MI khác không tự bị coi là nhiễu. Thành phần thấp có thể là drift/ST/T/chuyển động; được ghi để review, không tự high-pass. Rail 15,5 mV là cấu hình nghi ngờ gần giới hạn PTB danh nghĩa, không phải chuẩn chung.

Mỗi hàng hình là X/Y/Z, trục ngang Hz, trục dọc PSD mV²/Hz. Cột trái toàn dải, phải phóng to 50 Hz. PSD quanh 50 Hz cao chưa xác nhận nhiễu điện lưới.

In [ ]:
initial_quality=assess_input_quality(raw_xyz,fs,CFG)
print('Input QC:',initial_quality['flags'] or 'Chưa có lỗi dữ liệu rõ theo heuristic')
display(pd.DataFrame({'axis':list('XYZ'),
    'baseline_center_range_mV':initial_quality['baseline_center_range_mV'],
    'low_frequency_density':initial_quality['low_frequency_density'],
    'high_frequency_density':initial_quality['high_frequency_density']}))
f=initial_quality['psd_frequency_hz'];p=initial_quality['psd_density']
fig,axes=plt.subplots(3,2,figsize=(13,8),layout='constrained')
for j,name in enumerate('XYZ'):
    for col in range(2):
        axes[j,col].semilogy(f,np.maximum(p[:,j],1e-20),color=XYZ_COLORS[j])
        axes[j,col].axvline(CFG.notch_hz,color='#ea580c',ls=':')
        axes[j,col].set(xlim=(0,250) if col==0 else (42,58),ylabel=name+' PSD (mV²/Hz)')
        axes[j,col].grid(alpha=.2)
        if j==2:axes[j,col].set_xlabel('Tần số (Hz)')
fig.suptitle('XYZ gốc trước lọc: Welch toàn record; vùng 50 Hz cần thêm bằng chứng')
finish(fig)

## Bước 2 — FIR giữ băng thông và Notch có điều kiện

**Đầu vào:** XYZ gốc, được giữ nguyên; chất lượng và PSD được đánh giá trước lọc.

**Thao tác:** FIR pha tuyến tính 401 hệ số, Kaiser β=8, cutoff thiết kế 175 Hz tại fs=1000 Hz, áp dụng hai chiều. Mục tiêu kiểm chứng là giữ tốt đến 150 Hz và giảm thành phần trên 190–200 Hz. 175 Hz là điểm thiết kế, không có nghĩa biên độ mọi tần số đến 175 Hz được giữ nguyên.

Sau đó kiểm tra Notch theo từng trục: PSD Welch, độ nổi bật đỉnh hẹp và persistence theo các cửa sổ 4 giây. Có thêm fit harmonic trên các đoạn ít magnitude/ít slope, tránh coi mọi khoảng giữa QRS là yên. Timestamp của các đoạn rời rạc được giữ nguyên, không nối chúng rồi coi là tín hiệu liên tục.

- `noise_detected`: đủ bằng chứng theo heuristic; `auto` cho phép Notch riêng trục đó.
- `noise_not_detected`: không có đủ bằng chứng nghi nhiễu; giữ trục chưa Notch.
- `inconclusive`: thiếu/mâu thuẫn bằng chứng; giữ trục chưa Notch và gắn cờ review.

Ngưỡng 8 dB, persistence 60%, độ tập trung phổ và kiểm tra quiet đều là cấu hình nghiên cứu chưa được xác thực trên cohort. Vượt 8 dB một mình không bật Notch. Notch IIR Q=50 chạy hai chiều; `on/off` chỉ là chế độ thí nghiệm. Q=50 chưa được chứng minh tối ưu.

**Đầu ra:** morphology và nhánh Butterworth bậc 3, 5–30 Hz chỉ dùng detector. Không đo loop từ nhánh detector. Không tự thêm high-pass morphology; 0,5 Hz chỉ là tùy chọn đối chiếu với cảnh báo ST/T. FIR từ chối fs khác 1000 Hz thay vì âm thầm đổi thiết kế.

Hình ba cột giữ cách đọc cũ: gốc xám, morphology xanh, detector cam. Bảng RMS sai khác không tự là mức nhiễu đã loại; PSD/bảng loop phía dưới giúp kiểm tra đồng thời giảm thành phần nghi nhiễu và bảo toàn hình dạng.

In [ ]:
morph, detection_xyz, preprocessing = preprocess_xyz(raw_xyz, fs, CFG, return_diagnostics=True)
n_show = min(len(morph), int(4*fs))
record_time = np.arange(n_show)/fs

# Vẽ riêng ba tín hiệu để đường gốc không bị đường đã lọc che.
fig,axs = plt.subplots(3,3,figsize=(15,8),sharex=True,sharey='row',layout='constrained')
branches = [(raw_xyz,'.35','1. XYZ gốc'),
            (morph,'#2563eb','2. XYZ dùng cho beat/loop'),
            (detection_xyz,'#ea580c','3. XYZ chỉ dùng tìm nhịp')]
for j,name in enumerate('XYZ'):
    for col,(signal,color,title) in enumerate(branches):
        ax = axs[j,col]
        ax.plot(record_time,signal[:n_show,j],color=color,lw=1.1)
        ax.axhline(0,color='.75',lw=.7,ls=':')
        ax.grid(alpha=.2)
        if j == 0: ax.set_title(title)
        if col == 0: ax.set_ylabel(name+' (mV)')
        if j == 2: ax.set_xlabel('Thời gian recording (s)')
fig.suptitle('Ba cột riêng; cùng hàng dùng cùng thang điện thế')
finish(fig)

delta = morph[:n_show]-raw_xyz[:n_show]
display(pd.DataFrame({
    'Trục':list('XYZ'),
    'RMS sai lệch gốc–morph (µV)':np.sqrt(np.mean(delta**2,axis=0))*1000,
    'Biên độ max–min tín hiệu gốc (mV)':np.ptp(raw_xyz[:n_show],axis=0)
}))

### Bước 2A — Kiểm chứng đáp ứng FIR và Notch thực tế

**Đầu vào:** hệ số bộ lọc. **Thao tác:** `freqz` tính một chiều và bình phương gain để tính hai chiều. **Đầu ra:** gain tuyến tính và dB; bảng đo dải 0–150 Hz, stopband 190–200 Hz, cutoff và Notch. Hai chiều không trễ pha nhưng vẫn thay đổi biên độ/hiệu ứng mép.

Các con số này kiểm chứng bộ lọc tuyến tính, không chứng minh phân đoạn hay chẩn đoán đúng. Notch có thể làm mất cả thành phần tim thật gần 50 Hz; vì vậy cần quyết định có điều kiện.

In [ ]:
fig,response_table=plot_filter_responses(fs,CFG);finish(fig)
passband=response_table[response_table.frequency_hz<=CFG.fir_preserve_hz]
stopband=response_table[response_table.frequency_hz.between(*CFG.fir_stopband_hz)]
notch_band=response_table[response_table.notch_two_pass_db<=-3]
display(pd.DataFrame([
    {'measure':'FIR hai chiều: sai lệch gain lớn nhất 0–150 Hz','value':passband.fir_two_pass_db.abs().max(),'unit':'dB'},
    {'measure':'FIR hai chiều: gain cao nhất 190–200 Hz','value':stopband.fir_two_pass_db.max(),'unit':'dB'},
    {'measure':'Notch hai chiều: độ rộng mức ≤ −3 dB','value':notch_band.frequency_hz.max()-notch_band.frequency_hz.min(),'unit':'Hz'}]))
display(response_table[response_table.frequency_hz.isin([50,150,175,190,200])])

### Bước 2B — Vì sao bật hoặc không bật Notch

`P50` là mean PSD density của các bin 49,5–50,5 Hz; nền là median PSD density của các bin 45–48 và 52–55 Hz. Cả hai cùng đơn vị mV²/Hz, không so hai tích phân trên dải có độ rộng khác nhau. Welch dùng cửa sổ 4 s: độ phân giải 0,25 Hz ở dữ liệu đủ dài, không dùng zero-padding để giả tăng độ phân giải.

Fit coherent trên nhiều đoạn ít hoạt động là kiểm tra bổ sung; các đoạn này vẫn chỉ là ứng viên, không là PQ/TP đã được xác nhận. Nếu thiếu đủ số đoạn/độ dài/span, kết quả phổ mạnh chuyển `inconclusive`. Các hình theo thời gian là R_dB, không là xác suất nhiễu.

**Đầu ra dùng tiếp:** `notch_applied`, mode, các trục được lọc, status, score, persistence, reason và warnings. Giá trị phổ hậu lọc/cao tần còn lại là thành phần đo được, không tự có nghĩa “nhiễu còn lại”.

In [ ]:
print({k:preprocessing[k] for k in ['morphology_mode','notch_applied','notch_mode','notch_channels',
    'powerline_status','powerline_score_db','powerline_persistence','notch_reason']})
display(powerline_diagnostic_table(preprocessing).T)
print('Warnings:',preprocessing['warnings'])
print('Quiet intervals:',preprocessing['powerline'].get('quiet_info'))
for fig in plot_preprocessing_diagnostics(raw_xyz,morph,fs,preprocessing,CFG):finish(fig)

## Bước 3 — Gộp ba kênh thành đường hoạt động để tìm từng nhịp

**Đầu vào:** `detection_xyz`, ba kênh đã band-pass.

**Thao tác:** tính $E(t)=X_d(t)^2+Y_d(t)^2+Z_d(t)^2$, lấy trung bình trượt 80 ms rồi căn bậc hai. Kết quả là envelope RMS:

$$A(t)=\sqrt{\operatorname{mean}_{80\,ms} E(t)}.$$

Bình phương giúp phần dương/âm đều đóng góp vào hoạt động. Trung bình ngắn giúp các sườn Q/R/S gần nhau tạo một cụm hoạt động thay vì coi mỗi sườn là một nhịp.

**Đầu ra:** một đường 1D, luôn không âm, đơn vị mV. Đỉnh trên đường này là **ứng viên nhịp**, chưa phải biên QRS và không phải loop.

**80 ms là giá trị thử ban đầu, chưa được tối ưu trên PTB.** Phần “Bổ sung — Kiểm tra cửa sổ 80 ms bằng nhiều giá trị” cuối notebook có grid 20–200 ms, biểu đồ và tiêu chí đánh giá cụ thể.

In [ ]:
energy = np.sum(detection_xyz**2, axis=1)
window = ms_samples(CFG.energy_window_ms, fs)
envelope = np.sqrt(np.maximum(0, uniform_filter1d(energy, size=window)))
print('Cửa sổ trung bình:', window, 'mẫu =', window*1000/fs, 'ms')
fig,axs = plt.subplots(2,1,figsize=(12,6),sharex=True,layout='constrained')
axs[0].plot(record_time,energy[:n_show],color='#7c3aed');axs[0].set_ylabel('E(t) (mV²)')
axs[0].set_title('Tổng bình phương ba kênh detector')
axs[1].plot(record_time,envelope[:n_show],color='#2563eb');axs[1].set_ylabel('Envelope (mV)')
axs[1].set_title('Làm trung bình ngắn → mỗi cụm QRS thường tạo một đỉnh rõ')
axs[1].set_xlabel('Thời gian recording (s)')
for ax in axs:ax.grid(alpha=.2)
finish(fig)

## Bước 4 — Ngưỡng thích nghi chọn các đỉnh ứng viên

**Đầu vào:** envelope ở Bước 3.

**Thao tác:** trong vùng 2 giây quanh từng thời điểm, ước lượng nền bằng percentile 20 và mức cao bằng percentile 95. Ngưỡng bằng nền + 25% khoảng cách từ nền đến mức cao. Code ước lượng đường ngưỡng ở khoảng 100 Hz cho nhẹ, rồi nội suy về tần số gốc.

`find_peaks` chọn cực đại vượt ngưỡng, rộng ít nhất 15 ms và cách nhau ít nhất 250 ms. Các con số là tham số triển khai; không phải mọi tín hiệu bệnh lý đều đáp ứng chúng.

**Đầu ra:** `candidates`, chỉ số mẫu của các đỉnh ứng viên. Nếu T cao/nhiễu vẫn vượt ngưỡng, detector có thể chọn nhầm; cần kiểm tra bằng mắt trên XYZ.

In [ ]:
detector = detect_qrs_anchors(morph, detection_xyz, fs, CFG)
threshold = detector['threshold']
candidates, properties = find_peaks(envelope, height=threshold,
    distance=ms_samples(CFG.refractory_ms,fs), width=ms_samples(15,fs))
visible = candidates[candidates<n_show]
fig,ax = plt.subplots(figsize=(12,4),layout='constrained')
ax.plot(record_time,envelope[:n_show],color='#2563eb',label='Envelope')
ax.plot(record_time,threshold[:n_show],color='#dc2626',ls='--',label='Ngưỡng thích nghi')
ax.scatter(visible/fs,envelope[visible],color='#dc2626',s=35,label='Đỉnh ứng viên')
ax.set(xlabel='Thời gian recording (s)',ylabel='Envelope (mV)',title='Đỉnh vượt ngưỡng → ứng viên mốc QRS')
ax.legend();ax.grid(alpha=.2);finish(fig)
print('Ứng viên:', len(candidates), '| mốc sau tinh chỉnh/refractory:', len(detector['anchors']))

## Bước 5 — Tinh chỉnh một ứng viên thành mốc căn nhịp

**Đầu vào:** một đỉnh envelope. Trung bình trượt có thể làm đỉnh envelope lệch đỉnh điện thế.

**Thao tác:** xem XYZ morphology trong ±60 ms quanh ứng viên. Trừ baseline cục bộ lấy trước ứng viên rồi tìm điểm có magnitude lớn nhất trong vùng. Sau khi tinh chỉnh mọi ứng viên, áp dụng lại khoảng cách tối thiểu để tránh hai mốc quá gần nhau.

**Đầu ra:** một mốc QRS VCG cho từng nhịp. Mốc này dùng để căn các nhịp về cùng thời điểm. Nó không tự cho biết QRS bắt đầu/kết thúc; cũng không được coi là R trên ECG lead II.

In [ ]:
anchors = detector['anchors']
example_anchor = int(anchors[min(3,len(anchors)-2)])
p = int(candidates[np.argmin(np.abs(candidates-example_anchor))])
a,b = max(0,p-ms_samples(60,fs)), min(len(morph),p+ms_samples(60,fs)+1)
ba,bb = max(0,p-ms_samples(180,fs)), max(1,p-ms_samples(100,fs))
coarse_baseline = np.median(morph[ba:bb],axis=0)
local_mag = np.linalg.norm(morph[a:b]-coarse_baseline,axis=1)
refined_example = a+int(np.argmax(local_mag))
fig,axs = plt.subplots(2,1,figsize=(11,6),sharex=True,layout='constrained')
tt=(np.arange(a,b)-p)*1000/fs
axs[0].plot(tt,envelope[a:b],label='Envelope')
axs[1].plot(tt,local_mag,color='#7c3aed',label='Magnitude XYZ trừ baseline cục bộ')
for ax in axs:
    ax.axvline(0,color='#dc2626',ls='--',label='Ứng viên envelope')
    ax.axvline((refined_example-p)*1000/fs,color=QRS_COLOR,ls=':',label='Mốc QRS tinh chỉnh')
    ax.legend(fontsize=9);ax.grid(alpha=.2);ax.set_ylabel('mV')
axs[1].set_xlabel('Thời gian so với ứng viên envelope (ms)')
fig.suptitle('Chọn mốc trong vùng gần ứng viên; chưa chọn hai biên QRS')
finish(fig)
display(pd.DataFrame({'vai trò':['Ứng viên','Mốc tinh chỉnh','Mốc đã giữ trong detector'],
    'sample':[p,refined_example,example_anchor],
    'thời gian recording (s)':np.array([p,refined_example,example_anchor])/fs}))

## Bước 6 — Cắt một cửa sổ quanh mỗi mốc QRS

**Đầu vào:** morphology XYZ và danh sách mốc QRS.

**Thao tác:** tính median RR giữa các mốc. Khoảng trước mốc là min(280 ms, 40% RR); khoảng sau là min(650 ms, 78% RR). Những giới hạn này dùng để cắt beat, **chưa phải vùng QRS hoặc T**.

Loại beat sát biên recording, RR quá khác median hoặc có nguy cơ lẫn QRS kế tiếp. Đây là phân tích nhóm nhịp chiếm ưu thế; không đại diện đầy đủ cho ngoại tâm thu.

**Đầu ra:** các cửa sổ cùng chiều dài. Mỗi cửa sổ thường chứa P, QRS, ST và T; bước sau mới phân đoạn.

In [ ]:
rep = representative_beat(morph, anchors, fs, CFG, quality=preprocessing['input_quality'])
median_rr = rep['rr_seconds']
pre = rep['anchor']
post = len(rep['xyz'])-pre-1
print(f'Median RR: {median_rr*1000:.1f} ms')
print(f'Cửa sổ beat: {-pre*1000/fs:.1f} → {post*1000/fs:.1f} ms')
print('Beat được giữ:', rep['n_used'], '| loại:', rep['n_excluded'])
display(rep['beat_qc'].groupby('reason').size().rename('số beat').to_frame())
used = rep['beat_qc'].loc[rep['beat_qc']['reason']=='used','sample'].to_numpy(dtype=int)
example_anchor = int(used[min(2,len(used)-1)])
start,stop = example_anchor-pre,example_anchor+post+1
single = morph[start:stop].copy()
single_time=(np.arange(len(single))-pre)*1000/fs
fig,ax=plt.subplots(figsize=(12,4),layout='constrained')
draw_xyz(ax,single_time,single)
ax.axvline(0,color='#dc2626',ls='--',label='Mốc QRS VCG')
ax.set(xlabel='Thời gian so với mốc QRS (ms)',title='Đây là cả cửa sổ beat; chưa phải QRS loop')
ax.legend(ncol=4,loc='upper right');finish(fig)

## Bước 7 — Đặt gốc tọa độ bằng đoạn PQ phẳng

**Đầu vào:** một beat và mốc tại 0 ms.

**Thao tác:** tìm cửa sổ 25 ms ít thay đổi nhất trong vùng trước QRS. Lấy median X/Y/Z của cửa sổ đó thành vector baseline, rồi trừ vector này khỏi mọi điểm trong beat.

**Đầu ra:** các điểm được tịnh tiến về gốc tham chiếu PQ. Phép trừ vector hằng không thay đổi đạo hàm/tốc độ, nhưng thay đổi khoảng cách đến gốc, vì vậy ảnh hưởng `MaxVec`.

Vùng xám cần thực sự là PQ; “phẳng nhất” không bảo đảm đúng vị trí sinh lý. Drift trong một beat vẫn có thể còn lại.

QC bổ sung kiểm tra độ yên/tản mạn trong PQ; vùng phẳng nhất không tự là PQ đúng. Xem `pq_warning` ở `rep["beat_qc"]`. Offset PQ–TP được ghi trong `baseline_residual`, không tự coi là drift hoặc áp dụng high-pass.

In [ ]:
pq_single, baseline_single = flattest_pq(single,fs,pre,
    left_ms=-min(220,pre*1000/fs-5))
single_centered = single-baseline_single
display(pd.Series(baseline_single,index=['X baseline','Y baseline','Z baseline'],name='mV').to_frame())
fig,axs=plt.subplots(2,1,figsize=(12,6),sharex=True,layout='constrained')
for ax,signal,title in [(axs[0],single,'Trước trừ baseline PQ'),(axs[1],single_centered,'Sau trừ baseline PQ')]:
    draw_xyz(ax,single_time,signal)
    ax.axvspan(single_time[pq_single[0]],single_time[pq_single[1]-1],color='.4',alpha=.18,label='PQ 25 ms')
    ax.axhline(0,color='.4',ls=':',lw=1);ax.set_title(title);ax.legend(ncol=4)
axs[1].set_xlabel('Thời gian so với mốc QRS (ms)');finish(fig)

## Bước 8 — Gộp các beat thành một median beat

**Đầu vào:** những beat đã căn mốc và baseline.

**Thao tác:** so hình dạng vùng QRS với template median ban đầu; gắn cờ nhưng giữ beat có tương quan dưới 0,85 nếu không có lỗi dữ liệu rõ; hình thái khác có thể là thông tin MI. Tại mỗi thời điểm, lấy median riêng cho X, Y, Z của những beat được giữ. Không co giãn thời gian.

**Đầu ra:** `rep_xyz`, một beat đại diện. Đường đậm là median, các đường mờ là những beat đã giữ. Các biên QRS/T sau đây được tìm trên **beat đại diện này**, không phải độc lập trên mọi beat.

Có thể thấy QRS và T khác rõ về hình dạng trên XYZ khi không làm mượt quá dài. Median giảm ảnh hưởng ngoại lai nhưng không tự chứng minh mọi chi tiết là tín hiệu thật.

In [ ]:
rep_xyz = rep['xyz']; t = rep['time_ms']; anchor = rep['anchor']
fig,axs=plt.subplots(3,1,figsize=(12,8),sharex=True,layout='constrained')
for j,ax in enumerate(axs):
    for beat in rep['beats'][:20]: ax.plot(t,beat[:,j],color='.6',lw=.7,alpha=.20)
    ax.plot(t,rep_xyz[:,j],color=XYZ_COLORS[j],lw=2,label='Median '+'XYZ'[j])
    ax.axvline(0,color='.4',ls='--');ax.set_ylabel('XYZ'[j]+' (mV)');ax.legend();ax.grid(alpha=.2)
axs[-1].set_xlabel('Thời gian so với mốc QRS VCG (ms)')
fig.suptitle('Căn mốc → so hình dạng → median theo từng thời điểm')
finish(fig)
print('rep_xyz shape:', rep_xyz.shape, '| vị trí mốc 0 ms trong mảng:', anchor)

## Bước 9 — Tạo đường vận tốc để tìm biên QRS

**Đầu vào:** median XYZ.

**Thao tác:** tính $s(t)=\sqrt{(dX/dt)^2+(dY/dt)^2+(dZ/dt)^2}$ bằng đạo hàm SG ngắn 11 ms; trung bình thêm 5 ms cho đường dùng phân đoạn.

**Đầu ra:** tốc độ chuyển động của điểm điện thế XYZ, đơn vị mV/s. Đây không phải tốc độ dẫn truyền trong cơ tim.

Chú ý trong hình: QRS có thể tạo nhiều đỉnh vận tốc, tương ứng các sườn của Q/R/S. Đỉnh điện thế và đỉnh vận tốc không nhất thiết trùng nhau. Mục tiêu là tìm đầu/cuối **cả cụm hoạt động QRS**, không lấy riêng đỉnh cao nhất làm loop.

In [ ]:
boundaries = delineate_loops(rep,fs,CFG)
speed = boundaries['speed']; speed_q = boundaries['smooth_speed']
fig,axs=plt.subplots(2,1,figsize=(12,6),sharex=True,layout='constrained')
draw_xyz(axs[0],t,rep_xyz);axs[0].legend(ncol=3)
axs[1].plot(t,speed,color='.65',lw=.8,label='Tốc độ SG 11 ms')
axs[1].plot(t,speed_q,color='#4f46e5',label='Đường dùng phân đoạn: thêm mean 5 ms')
axs[1].set_ylabel('Tốc độ (mV/s)');axs[1].set_xlabel('Thời gian so với mốc QRS (ms)');axs[1].legend()
for ax in axs: ax.axvline(0,color='.4',ls='--');ax.grid(alpha=.2)
fig.suptitle('QRS thường đổi điện thế nhanh; vận tốc là đạo hàm của XYZ')
finish(fig)

## Bước 10 — Giới hạn vùng tìm QRS và đặt ngưỡng

**Đầu vào:** vận tốc và mốc tại 0 ms.

**Thao tác:** tìm QRS trong −180 đến +200 ms. Đỉnh vận tốc làm hạt giống được chọn trong vùng hẹp −80 đến +80 ms, để ưu tiên hoạt động gần mốc căn nhịp. Ngưỡng là max(nền PQ + 4 lần MAD chuẩn hóa, 7% đỉnh vận tốc).

**Đầu ra:** một ngưỡng và một đỉnh vận tốc gần mốc. Vùng xám chỉ là cửa sổ tìm kiếm; chưa phải QRS đã tìm được. Một đỉnh ở xa cửa sổ không được chọn chỉ vì cao hơn.

In [ ]:
qlo=max(0,anchor+int(CFG.qrs_search_ms[0]*fs/1000))
qhi=min(len(t)-1,anchor+int(CFG.qrs_search_ms[1]*fs/1000))
qpeak=boundaries['qrs_speed_peak']; qthreshold=boundaries['q_threshold']
pq0,pq1=rep['pq']
qbase=float(np.median(speed_q[pq0:pq1]));qnoise=robust_sigma(speed_q[pq0:pq1])
display(pd.DataFrame({'đại lượng':['Nền vận tốc PQ','MAD chuẩn hóa PQ','7% đỉnh','Ngưỡng được dùng','Thời điểm đỉnh vận tốc'],
    'giá trị':[qbase,qnoise,CFG.qrs_fraction*speed_q[qpeak],qthreshold,t[qpeak]],
    'đơn vị':['mV/s','mV/s','mV/s','mV/s','ms']}))
fig,ax=plt.subplots(figsize=(12,4),layout='constrained')
ax.plot(t,speed_q,color='#4f46e5',label='Vận tốc')
ax.axvspan(t[qlo],t[qhi],color='.5',alpha=.10,label='Vùng tìm QRS')
ax.axhline(qthreshold,color='#dc2626',ls='--',label='Ngưỡng')
ax.scatter(t[qpeak],speed_q[qpeak],color='#dc2626',s=55,label='Đỉnh hạt giống')
ax.axvline(0,color='.4',ls=':');ax.set(xlim=(-230,260),xlabel='Thời gian (ms)',ylabel='mV/s',title='Vùng tìm kiếm ≠ vùng QRS cuối cùng')
ax.legend();ax.grid(alpha=.2);finish(fig)

## Bước 11 — Tìm hai biên quanh cả cụm QRS

**Đầu vào:** các mẫu có vận tốc vượt ngưỡng.

**Thao tác:**

1. Tạo mask “trên ngưỡng”. Nối khe ngắn khoảng 15 ms để hạn chế cắt giữa hai sườn của cùng QRS.
2. Chọn cụm chứa đỉnh hạt giống ở Bước 10.
3. Từ hai đầu cụm, quét ra ngoài. Chỉ chốt biên khi có ít nhất 12 ms liên tục ở dưới ngưỡng.

**Đầu ra:** `qrs_on`, `qrs_off`. Một điểm tụt ngắn giữa hai đỉnh chưa đủ để làm biên. Nếu không có đoạn yên thì biên để thiếu, không dùng mép cửa sổ thay thế.

Hình đầu cho thấy việc nối khe; hình sau đánh dấu khoảng QRS cuối cùng.

In [ ]:
active_raw = speed_q[qlo:qhi+1] > qthreshold
active_closed = binary_closing(active_raw,structure=np.ones(ms_samples(15,fs),bool))
left=right=qpeak-qlo
while left>0 and active_closed[left-1]: left-=1
while right<len(active_closed)-1 and active_closed[right+1]: right+=1
selected_mask=np.zeros_like(active_closed);selected_mask[left:right+1]=True
q_on,q_off=boundaries['qrs_on'],boundaries['qrs_off']
fig,axs=plt.subplots(2,1,figsize=(12,6),sharex=True,layout='constrained')
axs[0].step(t[qlo:qhi+1],active_raw.astype(int),where='mid',color='.5',label='Mask ban đầu')
axs[0].step(t[qlo:qhi+1],active_closed.astype(int)+1.3,where='mid',color='#2563eb',label='Sau nối khe ngắn')
axs[0].step(t[qlo:qhi+1],selected_mask.astype(int)+2.6,where='mid',color=QRS_COLOR,label='Cụm đã chọn')
axs[0].set_yticks([.5,1.8,3.1],['Ban đầu','Nối khe','Cụm QRS']);axs[0].legend(loc='upper right',fontsize=9)
axs[1].plot(t,speed_q,color='#4f46e5');axs[1].axhline(qthreshold,color='#dc2626',ls='--',label='Ngưỡng')
if q_on is not None and q_off is not None:
    axs[1].axvspan(t[q_on],t[q_off],color=QRS_COLOR,alpha=.18,label=f'QRS: {t[q_on]:.0f} → {t[q_off]:.0f} ms')
    axs[1].axvline(t[q_on],color=QRS_COLOR,ls=':');axs[1].axvline(t[q_off],color=QRS_COLOR,ls=':')
axs[1].set(xlim=(-190,210),xlabel='Thời gian (ms)',ylabel='Vận tốc (mV/s)');axs[1].legend()
for ax in axs:ax.grid(alpha=.2)
fig.suptitle('Chọn cụm QRS, rồi tìm đoạn yên ở hai đầu')
finish(fig)

### Xem quá trình đếm đoạn yên từng mẫu

Bảng dưới cho thấy các lần quét ra ngoài cụm. `liên tiếp dưới ngưỡng` tăng khi mẫu thấp, trở về 0 khi mẫu cao. Khi đủ số mẫu, biên được trả về ở **mép gần QRS** của đoạn yên. Vì thế mốc trả về khác mẫu cuối cùng dùng để xác nhận đoạn yên.

In [ ]:
def trace_quiet(values,start,limit,direction,threshold,run):
    rows=[];count=0
    for i in range(start,limit+direction,direction):
        below=bool(values[i]<=threshold)
        count=count+1 if below else 0
        returned=None
        if count>=run:returned=i+run-1 if direction<0 else i-run+1
        rows.append({'sample':i,'thời gian (ms)':t[i],'vận tốc (mV/s)':values[i],
                     'dưới ngưỡng':below,'liên tiếp dưới ngưỡng':count,
                     'biên trả về (ms)':np.nan if returned is None else t[returned]})
        if returned is not None:break
    return pd.DataFrame(rows)

run=ms_samples(CFG.quiet_qrs_ms,fs)
on_trace=trace_quiet(speed_q,qlo+left,qlo,-1,qthreshold,run)
off_trace=trace_quiet(speed_q,qlo+right,qhi,1,qthreshold,run)
print('Cần',run,'mẫu liên tục =',run*1000/fs,'ms dưới ngưỡng.')
print('Quét trái: các bước cuối');display(on_trace.tail(run+2))
print('Quét phải: các bước cuối');display(off_trace.tail(run+2))

## Bước 12 — Tạo magnitude và tìm đỉnh T sau QRS

**Đầu vào:** median XYZ và QRS_off.

**Thao tác:** magnitude $M(t)=\sqrt{X^2+Y^2+Z^2}$ biểu thị khoảng cách của điểm XYZ đến gốc PQ. Làm mượt nhẹ magnitude 21 ms. Tìm đỉnh nổi bật trong vùng sau QRS, kết thúc trước nhịp kế tiếp theo median RR.

**Đầu ra:** một T peak candidate. Chọn đỉnh có prominence rõ trong vùng tìm kiếm, không lấy cực đại magnitude của toàn beat — cực đại toàn beat thường thuộc QRS.

Cụ thể, vùng tìm bắt đầu ở QRS_off + 20 ms và kết thúc ở min(600 ms, 70% median RR) sau mốc QRS. Đỉnh hợp lệ phải cách QRS_off ít nhất 50 ms và cách mép phải ít nhất 30 ms. Prominence tối thiểu là max(4 lần nhiễu magnitude PQ, 1,5% magnitude lớn nhất của QRS). Chọn prominence lớn nhất trong các đỉnh hợp lệ; lobe rõ khác trong ±100 ms được xét cùng và gắn cờ nhiều lobe.

Magnitude không giữ dấu của từng lead; sóng T âm trên Y vẫn có thể tạo đồi dương trên magnitude. Nhiều lobe T sẽ được gắn cờ để kiểm tra.

In [ ]:
mag = boundaries['magnitude']; mag_t=boundaries['t_magnitude']
tlo,thi = boundaries['t_search']; t_peak=boundaries['t_peak']
fig,axs=plt.subplots(2,1,figsize=(12,6),sharex=True,layout='constrained')
draw_xyz(axs[0],t,rep_xyz);axs[0].legend(ncol=3)
axs[1].plot(t,mag,color='.65',lw=.8,label='Magnitude')
axs[1].plot(t,mag_t,color='#0891b2',label='Magnitude làm mượt 21 ms')
if tlo is not None and thi is not None:
    axs[1].axvspan(t[tlo],t[thi],color=T_COLOR,alpha=.08,label='Vùng được phép tìm T')
if t_peak is not None:axs[1].scatter(t[t_peak],mag_t[t_peak],color=T_COLOR,s=60,label='Đỉnh T candidate')
axs[1].set_ylabel('Magnitude (mV)');axs[1].set_xlabel('Thời gian (ms)');axs[1].legend()
for ax in axs:ax.grid(alpha=.2)
fig.suptitle('XYZ có dấu → magnitude không âm → tìm T trong vùng sau QRS')
finish(fig)
if t_peak is None:print('Chưa tìm được T peak rõ; các bước tiếp tuyến sẽ không tạo biên giả.')
else:print(f'T peak candidate: {t[t_peak]:.1f} ms; magnitude: {mag_t[t_peak]:.4f} mV')

## Bước 13 — Xác định mức ST và baseline TP cho hai tiếp tuyến

**Đầu vào:** đường magnitude sau QRS.

**Thao tác:** lấy mức ST từ một vùng ngắn sau QRS. Sau T peak, tìm một đoạn TP phẳng 25 ms để ước lượng baseline cuối T. Dùng TP phía sau giúp hạn chế ảnh hưởng drift khi magnitude không về đúng mức PQ phía trước.

**Đầu ra:** hai mức điện thế tham chiếu, không phải hai biên. Mức ST dùng cho tiếp tuyến đi lên; baseline TP dùng cho tiếp tuyến đi xuống. XYZ dùng tính feature vẫn có gốc PQ chung, không tự bị tịnh tiến riêng theo TP.

Vùng PQ/TP tìm tự động cần kiểm tra. Mức ST khác PQ có thể do hình thái hoặc drift; cờ này không tự chẩn đoán ST elevation.

In [ ]:
tp=boundaries['tp']; st_level=boundaries['st_level']; tp_level=boundaries['t_end_baseline']
fig,ax=plt.subplots(figsize=(12,4),layout='constrained')
ax.plot(t,mag_t,color='#0891b2',label='Magnitude')
ax.axvspan(t[pq0],t[pq1-1],color='.5',alpha=.18,label='PQ đặt gốc XYZ')
if tlo is not None and t_peak is not None:
    st_stop=min(t_peak,tlo+ms_samples(25,fs))
    ax.axvspan(t[tlo],t[st_stop-1],color=T_COLOR,alpha=.15,label='Đoạn ước lượng mức ST')
if tp is not None:ax.axvspan(t[tp[0]],t[tp[1]-1],color='#7c3aed',alpha=.15,label='TP cho T-end')
if np.isfinite(st_level):ax.axhline(st_level,color=T_COLOR,ls='--',label='Mức ST')
if np.isfinite(tp_level):ax.axhline(tp_level,color='#7c3aed',ls=':',label='Baseline TP')
ax.set(xlabel='Thời gian (ms)',ylabel='Magnitude (mV)',title='Hai mức tham chiếu cho biên T')
ax.legend(fontsize=9,ncol=2);ax.grid(alpha=.2);finish(fig)
display(pd.DataFrame({'mức':['ST','TP cuối T'],'giá trị (mV)':[st_level,tp_level]}))

## Bước 14 — T-on từ tiếp tuyến sườn đi lên

**Đầu vào:** T peak, mức ST và magnitude làm mượt.

**Thao tác:** trước T peak, tìm điểm có độ dốc dương lớn nhất. Vẽ tiếp tuyến qua điểm đó; kéo tiếp tuyến về bên trái đến khi cắt mức ST.

**Đầu ra:** giao điểm là T-on candidate. Nó thường nằm trên chân sườn lên, không phải ngay T peak và không được cộng/trừ một padding cố định.

Nếu điểm tiếp tuyến là $(t_s,M_s)$ và độ dốc $k$ tính bằng mV/s:

$$t_{on}=t_s+\frac{M_{ST}-M_s}{k}.$$

Tất cả thời gian trong công thức phải cùng đơn vị; code đổi sang chỉ số mẫu rồi hiển thị ms. Nếu giao điểm không hợp lệ, dùng candidate theo ngưỡng nếu có và gắn cờ. **Đây là định nghĩa thao tác của T-on**, không phải điểm bắt đầu T đã được xác nhận lâm sàng.

In [ ]:
rise=boundaries['t_rise_index']; tangent_on=boundaries['t_tangent_on']
fig,ax=plt.subplots(figsize=(12,4),layout='constrained')
ax.plot(t,mag_t,color='#0891b2',label='Magnitude')
if tlo is not None and thi is not None:ax.set_xlim(t[tlo]-10,t[thi]+10)
if t_peak is not None:ax.set_ylim(-.02,max(.05,mag_t[t_peak]*1.3))
if rise is not None and tangent_on is not None:
    slope=boundaries['t_slope'][rise]
    ix=np.array([tangent_on,rise])
    yy=mag_t[rise]+slope*(ix-rise)/fs
    ax.plot(t[ix],yy,color=T_COLOR,ls='-.',lw=2,label='Tiếp tuyến sườn lên')
    ax.scatter(t[rise],mag_t[rise],color=T_COLOR,s=50,label='Điểm dốc lên lớn nhất')
    ax.axhline(st_level,color='.5',ls='--',label='Mức ST')
    ax.axvline(t[tangent_on],color=T_COLOR,ls=':',label=f'T-on = {t[tangent_on]:.0f} ms')
    ax.scatter(t[tangent_on],st_level,color=T_COLOR,marker='x',s=80)
    display(pd.DataFrame({'giá trị':['Điểm tiếp tuyến (ms)','Magnitude tại điểm (mV)','Độ dốc (mV/s)','Mức ST (mV)','Giao điểm T-on (ms)'],
                         'số':[t[rise],mag_t[rise],slope,st_level,t[tangent_on]]}))
else:print('Không có giao điểm tiếp tuyến T-on hợp lệ; xem cờ QC.')
ax.set(xlabel='Thời gian (ms)',ylabel='Magnitude (mV)',title='Kéo tiếp tuyến sườn lên về mức ST → T-on')
ax.legend(fontsize=9);ax.grid(alpha=.2);finish(fig)

## Bước 15 — T-off từ tiếp tuyến sườn đi xuống

**Đầu vào:** T peak và baseline TP.

**Thao tác:** sau T peak (hoặc lobe cuối nếu T nhiều lobe), tìm điểm có độ dốc âm lớn nhất về trị tuyệt đối. Kéo tiếp tuyến xuống tới mức TP.

**Đầu ra:** giao điểm là T-off candidate. Đây là thời điểm kết thúc theo quy tắc tiếp tuyến, có thể sớm hơn thời điểm magnitude gần như về hẳn baseline.

Trong hình, đường tím là tiếp tuyến; đường xám đứng là T-end ước lượng bằng ngưỡng. Bất đồng lớn hoặc không đủ vùng TP sẽ gắn cờ. Code không mặc định lấy mép cửa sổ tìm kiếm làm T-off.

In [ ]:
fall=boundaries['t_fall_index']; tangent_off=boundaries['t_tangent_off']
threshold_off=boundaries['t_threshold_off']
fig,ax=plt.subplots(figsize=(12,4),layout='constrained')
ax.plot(t,mag_t,color='#0891b2',label='Magnitude')
if tlo is not None and thi is not None:ax.set_xlim(t[tlo]-10,t[thi]+10)
if t_peak is not None:ax.set_ylim(-.02,max(.05,mag_t[t_peak]*1.3))
if fall is not None and tangent_off is not None:
    slope=boundaries['t_slope'][fall]
    ix=np.array([fall,tangent_off]);yy=mag_t[fall]+slope*(ix-fall)/fs
    ax.plot(t[ix],yy,color='#7c3aed',ls='-.',lw=2,label='Tiếp tuyến sườn xuống')
    ax.scatter(t[fall],mag_t[fall],color='#7c3aed',s=50,label='Điểm dốc xuống mạnh nhất')
    ax.axhline(tp_level,color='.5',ls='--',label='Baseline TP')
    ax.axvline(t[tangent_off],color=T_COLOR,ls=':',label=f'T-off tangent = {t[tangent_off]:.0f} ms')
    ax.scatter(t[tangent_off],tp_level,color=T_COLOR,marker='x',s=80)
    display(pd.DataFrame({'giá trị':['Điểm tiếp tuyến (ms)','Magnitude tại điểm (mV)','Độ dốc (mV/s)','Baseline TP (mV)','Giao điểm T-off (ms)'],
                         'số':[t[fall],mag_t[fall],slope,tp_level,t[tangent_off]]}))
else:print('Không có giao điểm tiếp tuyến T-off hợp lệ; xem cờ QC.')
if threshold_off is not None:ax.axvline(t[threshold_off],color='.5',ls='--',label=f'T-off theo ngưỡng = {t[threshold_off]:.0f} ms')
ax.set(xlabel='Thời gian (ms)',ylabel='Magnitude (mV)',title='Kéo tiếp tuyến sườn xuống về baseline TP → T-off')
ax.legend(fontsize=9);ax.grid(alpha=.2);finish(fig)

## Bước 16 — Đặt bốn mốc lên cùng một biểu đồ XYZ

**Đầu vào:** median XYZ và bốn biên vừa ước lượng.

**Thao tác:** tô hai khoảng thời gian. Xanh từ QRS_on đến QRS_off; cam từ T_on đến T_off. Không lấy đoạn giữa hai khoảng làm QRS hay T.

**Đầu ra:** biểu đồ kiểm tra tổng hợp. Trên HC demo, bạn nên thấy cụm QRS nhọn trong vùng xanh, T rộng hơn trong vùng cam. T hướng âm trên một trục vẫn thuộc cùng đoạn T.

Một vùng tô màu chỉ biểu diễn mốc thuật toán chọn. Nếu cắt mất sườn QRS hoặc lấy nhầm P/T, phải xem lại detector/biên; không chỉ nhìn màu rồi kết luận thành công.

In [ ]:
names=['qrs_on','qrs_off','t_on','t_peak','t_off']
boundary_table=pd.DataFrame([{'mốc':name,'chỉ số trong median beat':boundaries[name],
    'thời gian (ms)':np.nan if boundaries[name] is None else t[boundaries[name]]} for name in names])
display(boundary_table)
fig,ax=plt.subplots(figsize=(12,5),layout='constrained')
draw_xyz(ax,t,rep_xyz);shade(ax,t,boundaries)
ax.axvline(0,color='.4',ls='--',label='Mốc QRS tại 0 ms')
ax.set(xlabel='Thời gian so với mốc QRS (ms)',title='Bốn mốc → hai đoạn được chọn trên XYZ')
ax.legend(ncol=3,fontsize=9);finish(fig)
print('Cờ phân đoạn:', boundaries['flags'] or 'Không có cờ heuristic; vẫn cần xem bằng mắt.')

## Bước 17 — Cắt mảng thật sự: từ hai vùng màu thành hai tập điểm

**Đầu vào:** `rep_xyz` và các chỉ số biên. Thời gian ms dùng để đọc hình; Python slice dùng **chỉ số mẫu**.

**Thao tác:**

```python
qrs_loop = rep_xyz[qrs_on:qrs_off + 1]
t_loop   = rep_xyz[t_on:t_off + 1]
```

**Đầu ra:** mỗi loop vẫn có ba cột X/Y/Z. Chỉ số tương đối trong median beat khác chỉ số tuyệt đối của recording. Ví dụ, ở fs=1000 Hz và anchor=280, chỉ số 249 ứng với −31 ms, không phải 249 ms sau R.

`+1` giữ cả mẫu cuối vì điểm dừng của Python slice không được lấy. Nếu thiếu biên hoặc thứ tự sai, loop để `None`, không tạo một đoạn giả.

In [ ]:
qrs_loop = cut_loop(rep_xyz,boundaries['qrs_on'],boundaries['qrs_off'])
t_loop = cut_loop(rep_xyz,boundaries['t_on'],boundaries['t_off'])
for name,loop,on,off in [('QRS',qrs_loop,'qrs_on','qrs_off'),('T',t_loop,'t_on','t_off')]:
    if loop is None:
        print(name,': chưa có biên hợp lệ');continue
    lo,hi=boundaries[on],boundaries[off]
    print(f'{name}: slice [{lo}:{hi+1}], shape {loop.shape}; {t[lo]:.0f} → {t[hi]:.0f} ms')
    ix=np.unique(np.r_[np.arange(min(3,len(loop))),np.arange(max(0,len(loop)-3),len(loop))])
    display(pd.DataFrame({'hàng trong loop':ix,'sample median beat':lo+ix,'thời gian (ms)':t[lo+ix],
                          'X (mV)':loop[ix,0],'Y (mV)':loop[ix,1],'Z (mV)':loop[ix,2]}))

## Bước 18 — Nối các hàng XYZ theo thời gian thành loop 3D

**Đầu vào:** hai mảng vừa cắt. Mỗi hàng là một điểm trong không gian điện thế.

**Thao tác:** nối hàng 0 → hàng 1 → hàng 2 → … theo đúng thứ tự. Không sắp xếp điểm theo X/Y/Z vì sẽ phá quỹ đạo thời gian.

**Đầu ra:** QRS loop và T loop 3D. Chấm tròn là điểm đầu, dấu × là điểm cuối; chúng không buộc trùng nhau nên loop đo được có thể chưa kín hoàn toàn.

Hai hình dùng cùng giới hạn trục để không phóng đại loop nhỏ hơn. Một đường 3D nhìn trên màn hình có thể chồng nét do góc nhìn; không có nghĩa các thời điểm giống nhau.

In [ ]:
available=[p for p in [qrs_loop,t_loop] if p is not None]
fig=plt.figure(figsize=(13,6))
fig.subplots_adjust(left=.04,right=.92,wspace=.18,top=.85,bottom=.12)
if available:
    points=np.vstack(available);low=points.min(axis=0);high=points.max(axis=0)
    center=(low+high)/2;radius=max((high-low).max()/2,1e-3)*1.15
for pos,loop,color,name in [(1,qrs_loop,QRS_COLOR,'QRS'),(2,t_loop,T_COLOR,'T')]:
    ax=fig.add_subplot(1,2,pos,projection='3d')
    if loop is not None:
        ax.plot(loop[:,0],loop[:,1],loop[:,2],color=color,lw=2)
        ax.scatter(*loop[0],color=color,s=55,marker='o',label='Đầu')
        ax.scatter(*loop[-1],color=color,s=60,marker='x',label='Cuối')
        ax.legend();ax.set_xlim(center[0]-radius,center[0]+radius);ax.set_ylim(center[1]-radius,center[1]+radius);ax.set_zlim(center[2]-radius,center[2]+radius)
        ax.set_box_aspect((1,1,1))
    ax.set(xlabel='X (mV)',ylabel='Y (mV)',zlabel='',title=name+' loop 3D')
    ax.text2D(.93,.58,'Z (mV)',transform=ax.transAxes,rotation=90,ha='center',va='center')
fig.suptitle('Hai đoạn thời gian → hai quỹ đạo XYZ')
finish(fig)

## Bước 19 — Chiếu X–Y và PCA là hai thao tác khác nhau

**Đầu vào:** loop 3D đã tách; bước chiếu không tìm lại biên.

**Chiếu Frank X–Y:** lấy hai cột X/Y, bỏ Z. Vị trí tương đối QRS/T vẫn cùng hệ tọa độ PQ; vòng có thể nhìn hẹp nếu biến thiên nhiều ở Z.

**PCA riêng:** trừ tâm của loop, chọn hai hướng vuông góc giữ nhiều biến thiên nhất rồi chiếu xuống mặt phẳng đó. Dùng PCA để tính Arc/Area. PC1/PC2 không phải X/Y. Mỗi loop có hệ PCA riêng, vì vậy không dùng hai hình PCA để đo vị trí tương đối hoặc góc QRS–T.

**Đầu ra:** mảng `(n,2)` của từng loop. Trong hình, XYZ và X–Y dùng gốc chung; hai hình PCA đặt riêng. Phương sai giữ lại mô tả phép chiếu, không phải độ chính xác phân đoạn.

In [ ]:
plot_result={'qrs_loop':qrs_loop,'t_loop':t_loop}
finish(plot_loops(plot_result))
for name,loop in [('QRS',qrs_loop),('T',t_loop)]:
    projection=project_pca(loop)
    if projection is None:continue
    model=projection['model']
    print(name,'tâm XYZ trước PCA:',np.round(model.mean_,4))
    display(pd.DataFrame(model.components_,index=['PC1','PC2'],columns=['hệ số X','hệ số Y','hệ số Z']))
    print('Shape sau chiếu:',projection['xy'].shape,'| variance:',round(projection['variance']*100,2),'%')

## Bước 20 — Từ hai loop đến 12 đặc trưng

**Đầu vào:** hai loop XYZ, hình chiếu PCA và tốc độ trên toàn median beat.

**Thao tác:**

| Đặc trưng | Đo gì | Dữ liệu | Đơn vị |
|---|---|---|---|
| Arc | Tổng độ dài đoạn nối theo thời gian | PCA 2D | mV |
| Area | Diện tích shoelace, đóng cuối về đầu | PCA 2D | mV² |
| MaxVec | Điểm xa gốc PQ nhất | XYZ | mV |
| MaxCentroidDistance | Điểm xa tâm loop nhất | XYZ, chỉ QRS | mV |
| Max/Mean/StdVel | Tốc độ lớn nhất/trung bình/biến thiên | Đạo hàm XYZ | mV/s |

**Đầu ra:** 7 feature QRS + 5 feature T cho một recording. QRS có Arc và MaxCentroidDistance; T không có hai feature này trong bộ 12.

Area không phải diện tích mô tim. Với quỹ đạo tự cắt, shoelace cho trị tuyệt đối diện tích đại số, không bảo đảm bằng tổng mọi vùng tô. Arc ở đây không thêm đoạn đóng vòng. Vận tốc lấy trên toàn median beat rồi mới cắt, tránh tính đạo hàm độc lập tại mép mỗi loop.

In [ ]:
loop_speeds={name:None if loop is None else speed[boundaries[on]:boundaries[off]+1]
    for name,loop,on,off in [('QRS',qrs_loop,'qrs_on','qrs_off'),('T',t_loop,'t_on','t_off')]}
features = extract_features(qrs_loop,t_loop,fs,CFG,speeds=loop_speeds)
display(pd.DataFrame([{'feature':name,'giá trị candidate':features[name],
    'đơn vị':'mV²' if name.endswith('Area') else ('mV/s' if name.endswith('Vel') else 'mV')}
    for name in FEATURE_NAMES]))

## Bước 21 — Kiểm tra kết quả trước khi coi là feature dùng được

**Đầu vào:** toàn bộ detector, median beat và biên. Có thể tính được feature vẫn chưa đồng nghĩa đã cắt đúng đoạn.

**Thao tác:** kiểm tra RR, số beat, tỷ lệ loại beat, thứ tự/duration của biên, mức ST, độ yên TP, T nhiều lobe và bất đồng tiếp tuyến/ngưỡng. Bảng bên dưới là QC của chính record đang theo dõi.

**Đầu ra:** `auto_candidate`, `needs_review`, hoặc `manual_reviewed` nếu có biên đã sửa bằng tay. `auto_candidate` chỉ có nghĩa chưa vi phạm các quy tắc được cài đặt, không có nghĩa đã khớp annotation chuẩn.

| Cờ thường gặp | Bạn cần xem gì |
|---|---|
| `st_level_requires_review` | Mức ST khác PQ; xem lại vị trí T-on và baseline |
| `t_end_methods_disagree` | Tiếp tuyến và ngưỡng chọn cuối T khác nhiều |
| `t_off_no_baseline_return` | Không tìm được đoạn về nền rõ trong vùng T |
| `qrs_on/off_not_found` | Không có đoạn yên đủ dài để xác định biên QRS |
| `rr_irregular`, `possible_missed_beat` | Detector hoặc nhịp không đều; xem recording |
| `features_incomplete` | Có loop thiếu/không đủ mẫu; không ép đủ 12 số |

Đổi record MI ở ô cấu hình rồi Run All sẽ cho thấy cùng thuật toán có thể gặp hình thái khác. Cần kiểm tra lại tất cả hình, không chỉ bảng feature.

In [ ]:
complete_result=process_record(DATA_ROOT/RECORD_ID,CFG)
display(pd.DataFrame([qc_row(complete_result)]))
print('Cờ QC:',complete_result['flags'] or 'Không có cờ heuristic')
print('Qua QC nội bộ:',complete_result['eligible_for_features'])
print('Tổng hợp: nhịp phát hiện',len(anchors),'; beat giữ',rep['n_used'],
      '; QRS samples',None if qrs_loop is None else len(qrs_loop),
      '; T samples',None if t_loop is None else len(t_loop))

## Bước 22 — So sánh với một ca MI trên cùng cách xử lý

Ô này minh họa vì sao ca bệnh lý cần QC ngay cả khi code vẫn tạo được hai vùng màu. Đây chỉ là một ví dụ; không dùng hai record để kết luận hiệu quả phân loại.

Xem vùng xanh có bao cả các sườn QRS không, vùng cam bắt đầu/kết thúc ở đâu so với hình sóng T, và mức ST/PQ. Nếu có cờ ST, T-on theo tiếp tuyến cần xem lại; feature vẫn là candidate.

In [ ]:
COMPARISON_RECORD = 'patient001/s0010_re'
comparison=process_record(DATA_ROOT/COMPARISON_RECORD,CFG)
display(pd.DataFrame([qc_row(complete_result),qc_row(comparison)]))
finish(plot_delineation(comparison))

## Bản đồ toàn bộ thao tác

| Sau bước nào | Đối tượng đang có | Đã là loop chưa? |
|---|---|---|
| Đọc/lọc | Recording XYZ nhiều nhịp | Chưa |
| Detector | Danh sách thời điểm QRS | Chưa |
| Cắt beat | Nhiều cửa sổ quanh các mốc | Chưa |
| Median | Một beat đại diện | Chưa |
| Phân đoạn | Bốn chỉ số QRS_on/off, T_on/off | Chưa cắt mảng |
| Cắt XYZ | Hai mảng `(n_QRS,3)` và `(n_T,3)` | Có |
| Nối/vẽ | Hai quỹ đạo theo thứ tự thời gian | Có |
| PCA/feature | Hình chiếu và 12 con số | Loop đã được tạo từ trước |

Mấu chốt: **vận tốc và magnitude giúp tìm bốn mốc; XYZ trong hai khoảng đó mới tạo thành hai loop.**

Các hàm nền dùng chung notebook 04, với tiền xử lý có điều kiện và QC giữ hình thái khác thường. Xem nguồn và phạm vi kiểm chứng ở notebook 04: detector energy/speed và các ngưỡng cụ thể là implementation của project, không phải bản sao một thuật toán đã có số liệu validation. Walkthrough theo một record; các mục kiểm chứng riêng chạy thí nghiệm và lưu kết quả vào thư mục riêng. Không ghi đè vcg_12_features.csv hoặc thay logic ML/nhãn của pipeline cũ.

## Đánh giá bộ lọc — A cũ, B FIR/auto, C chưa lọc bổ sung

A: Butterworth 80 Hz + Notch 50 Hz Q=30 luôn bật. B: FIR 401/Kaiser8/cutoff175 + Notch auto Q=50. C: không low-pass/Notch bổ sung. Cả ba dùng cùng detector/phân đoạn/công thức feature và QC mới; A tái hiện bộ lọc cũ, không tái hiện việc loại mọi beat khác hình thái.

**Hai phép kiểm chứng:**

1. `fixed`: dùng cùng anchors, tập beat của B, vùng PQ của từng beat, PQ cuối của median và bốn biên B. Như vậy thay đổi feature chủ yếu phản ánh XYZ sau lọc, không lẫn việc chọn lại mốc/PQ.
2. `full`: chạy lại A/B/C; anchors, beat/PQ và biên có thể khác. Bảng thời gian tính theo mốc căn nhịp của từng nhánh, nên phải đọc cùng số mốc và số beat.

**Hình:** XYZ mV/tốc độ mV/s theo ms; loops 3D có tọa độ điện thế mV, dùng chung thang A/B/C trong mỗi ô. Vùng tô là biên B để tham chiếu. **Bảng feature:** giá trị, sai khác tuyệt đối và phần trăm so với C; khi C gần 0 hoặc thiếu dữ liệu, % để NaN. Gần C không tự chứng minh đúng vì C có thể còn nhiễu.

In [ ]:
preprocessing_comparisons={}
for comparison_record in [RECORD_ID,'patient001/s0010_re']:
    if comparison_record in preprocessing_comparisons:continue
    comparison=compare_preprocessing_configs(DATA_ROOT/comparison_record,CFG)
    preprocessing_comparisons[comparison_record]=comparison
    print(comparison_record,'| errors:',comparison['errors'])
    keep=['config','n_qrs_detected','n_beats_used','qrs_on_ms','qrs_off_ms','t_on_ms','t_off_ms',
          'notch_applied','powerline_status','flags']
    display(comparison['full_qc'][keep].set_index('config').T)
    display(comparison['signal_metrics'])
    for kind in ['fixed','full']:
        print('Comparison:',kind)
        finish(plot_median_config_comparison(comparison,kind))
        finish(plot_3d_config_comparison(comparison,kind))
        values=comparison['feature_values'].query('comparison == @kind')
        display(values.pivot(index='feature',columns='config',values='value').reindex(FEATURE_NAMES))
        display(comparison['feature_changes'].query('comparison == @kind and config != "C_unfiltered"')
                [['config','feature','absolute_change','relative_change_pct']])
output_dir=Path('vcg_preprocessing_results');output_dir.mkdir(exist_ok=True)
for record,comparison in preprocessing_comparisons.items():
    label=record.replace('/','_')
    comparison['full_qc'].to_csv(output_dir/(label+'_qc.csv'),index=False)
    comparison['feature_changes'].to_csv(output_dir/(label+'_feature_changes.csv'),index=False)
    comparison['signal_metrics'].to_csv(output_dir/(label+'_signal_metrics.csv'),index=False)
print('Kết quả PTB thật lưu tại:',output_dir.resolve())

## Kiểm thử tiền xử lý và giới hạn

Các tests A–F trong `tests/test_vcg_preprocessing.py` dùng tín hiệu tổng hợp được ghi rõ là tổng hợp: sạch; sin 50 Hz liên tục; packet 50 Hz ở QRS; không ổn định/thiếu dữ liệu; đáp ứng FIR/Notch hai chiều; shape, NaN, biên và không có beat. Có thêm kiểm tra mode và chỉ Notch riêng trục nghi nhiễu, dữ liệu lỗi, giữ hợp đồng hai mảng và smoke HC/MI.

FIR 401/Kaiser8/cutoff175, Notch Q50, ngưỡng 8 dB/60%, quiet mask và baseline QC đều là lựa chọn nghiên cứu. Các tests này không xác nhận ngưỡng tối ưu trên bệnh nhân. Quiet mask có thể nhầm đoạn ST/T ít dốc; khi không đủ bằng chứng, cần review. High-pass 0,5 Hz không mặc định bật. Feature thay đổi cần đọc cả mức giảm thành phần nghi nhiễu và mức đổi QRS/ST/T.

Thí nghiệm `energy_window_ms` bên dưới giữ nguyên grid/detector/seed và đóng băng bộ lọc cũ qua `WINDOW_PREPROCESSING_CFG=legacy_config(CFG)` để không trộn hai nghiên cứu. 80 **ms** là thời gian làm envelope; 80 **Hz** là cutoff cũ. Dữ liệu tổng hợp 250/500 Hz dùng legacy trong nghiên cứu đó; FIR mới mặc định yêu cầu 1000 Hz.

In [ ]:
import unittest
suite=unittest.defaultTestLoader.discover('tests',pattern='test_vcg_preprocessing.py')
test_result=unittest.TextTestRunner(verbosity=2).run(suite)
if not test_result.wasSuccessful():raise RuntimeError('Preprocessing tests failed; không công bố kết quả như đã qua kiểm tra.')

### Nguồn và phần do project đề xuất

- [Kligfield et al. 2007](https://www.jacc.org/doi/10.1016/j.jacc.2007.01.024): tham khảo bảo toàn băng thông ECG chẩn đoán; không chứng minh FIR 401/cutoff175 tối ưu cho VCG.
- [Kijonka et al. 2022](https://www.frontiersin.org/journals/physiology/articles/10.3389/fphys.2022.941827/full): paper dùng FIR high-pass và Notch khi có powerline, cùng thuật toán wavelet/PQ. Project giữ PQ và không mặc định dùng high-pass đó.
- [Representative QRS loop, volume 2023, công bố 2024](https://www.frontiersin.org/journals/physiology/articles/10.3389/fphys.2023.1260074/full): tham khảo auto-Notch và bảo toàn ST/T; paper dùng thiết kế FIR khác, không dùng nguyên bộ PSD 8 dB/60% hoặc IIR Q50 ở đây.
- [SciPy firwin](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.firwin.html), [filtfilt](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.filtfilt.html): cutoff firwin là half-amplitude; hai chiều có gain |H|². Những con số thực tế ở trên được tính từ hệ số của project.

Notebook 04/05 cùng import `vcg_preprocessing.py`, `vcg_pipeline.py` và `vcg_preprocessing_evaluation.py`; không sao chép hai engine khác nhau. Các notebook 02/03 và script cũ vẫn là phiên bản lịch sử, cần migration riêng trước khi trộn feature/ML. Không thay nhãn HC/MI hoặc công thức 12 feature.

## Bổ sung — Kiểm tra cửa sổ 80 ms bằng nhiều giá trị

Phần này thử **20, 40, 60, 80, 100, 120, 150, 180 và 200 ms** thay vì mặc định coi 80 ms là tối ưu. Giữ nguyên mọi tham số khác; chỉ thay `energy_window_ms`. Phần này đánh giá **phát hiện mốc nhịp**, chưa đánh giá độ chính xác QRS_on/off và T_on/off.

Hai nguồn bằng chứng được tách riêng:

1. **PTB thật:** 30 giây đầu của mọi record MI/HC. Workspace không có annotation nhịp chuẩn được dùng ở đây, nên chỉ đo số mốc, cờ RR và độ đồng thuận giữa các cửa sổ. Đồng thuận không phải độ chính xác.
2. **XYZ tổng hợp:** có đồng hồ nhịp được tạo trước khi chạy detector, nên đo được TP/FP/FN và F1. Đây là mô hình đơn giản, không thay thế validation trên người thật.

Các hình/bảng bên dưới được tính bằng code và lưu sẵn. Không dùng đầu ra 80 ms làm ground truth. Không sửa tín hiệu tổng hợp, tolerance hoặc chọn seed sau khi xem kết quả để ưu tiên 80 ms.

Bộ lọc của nghiên cứu cửa sổ được giữ ở cấu hình A cũ bằng `legacy_config`, độc lập với phần đánh giá FIR mới. Các kết quả detector trong mục này không được dùng để khẳng định tiền xử lý mới đã tối ưu.

### A. Cấu hình phép thử và hàm đánh giá

Tolerance ghép mốc ở phép thử tổng hợp là **50 ms**, ghép một-một theo thời gian. Prediction xa mọi mốc hơn tolerance là FP; mốc không có prediction ghép là FN. F1 = 2TP/(2TP+FP+FN). MAE chỉ tính trên các mốc ghép được, nên phải đọc cùng FP/FN.

Tín hiệu tổng hợp gồm các xung Gaussian Q/R/S/P/T cộng baseline wander, nhiễu trắng và nhiễu 50 Hz, với 5 điều kiện: sạch, nhiễu, T lớn, QRS rộng, RR không đều. Các độ rộng/biên độ được sinh từ khoảng đã đặt trong generator, không từ 80 ms. Các condition là kịch bản kỹ thuật, không phải chẩn đoán.

Có 45 subject tổng hợp, mỗi subject có 2 recording 20 giây. Chia 30 subject cho validation và 15 cho holdout; các record cùng subject chỉ ở một split. Chọn cửa sổ bằng F1 validation, hòa thì MAE thấp hơn, tiếp tục hòa thì cửa sổ nhỏ hơn. Không đổi lựa chọn sau khi xem holdout. Bỏ 1 giây ở mỗi biên cho cả prediction và reference để giảm transient lọc.

In [ ]:
from dataclasses import replace
WINDOW_PREPROCESSING_CFG=legacy_config(CFG)

WINDOWS_MS = [20, 40, 60, 80, 100, 120, 150, 180, 200]
WINDOW_BENCHMARK_SEED = 20261010
MATCH_TOLERANCE_MS = 50.0


def match_qrs_events(predicted_s, reference_s, tolerance_ms=50.0):
    """Ghép một-một theo thời gian; không dùng detector 80 ms làm nhãn chuẩn."""
    predicted_s = np.sort(np.asarray(predicted_s, float))
    reference_s = np.sort(np.asarray(reference_s, float))
    tolerance = tolerance_ms / 1000.0
    i = j = 0
    errors = []
    while i < len(predicted_s) and j < len(reference_s):
        delta = predicted_s[i] - reference_s[j]
        if abs(delta) <= tolerance + 1e-12:
            errors.append(abs(delta) * 1000.0)
            i += 1; j += 1
        elif delta < -tolerance:
            i += 1
        else:
            j += 1
    tp = len(errors)
    return {'tp':tp, 'fp':len(predicted_s)-tp, 'fn':len(reference_s)-tp,
            'error_sum_ms':float(np.sum(errors)), 'n_matched':tp}


def summarize_event_scores(frame, groupby):
    result = frame.groupby(groupby, dropna=False)[['tp','fp','fn','error_sum_ms','n_matched']].sum().reset_index()
    result['precision'] = result['tp']/(result['tp']+result['fp']).replace(0,np.nan)
    result['recall'] = result['tp']/(result['tp']+result['fn']).replace(0,np.nan)
    result['f1'] = 2*result['tp']/(2*result['tp']+result['fp']+result['fn']).replace(0,np.nan)
    result['matched_mae_ms'] = result['error_sum_ms']/result['n_matched'].replace(0,np.nan)
    return result


def generate_xyz_case(subject, repetition, duration=20.0):
    """Mô hình tổng hợp đơn giản, không phải mô phỏng sinh lý/clinical benchmark.

    Tất cả cửa sổ nhận đúng cùng arrays; đồng hồ QRS được tạo trước detector.
    Profile/seed không phụ thuộc cửa sổ đang thử hoặc mục tiêu ưu tiên 80 ms.
    """
    profile_rng = np.random.default_rng(WINDOW_BENCHMARK_SEED + subject*1009)
    rng = np.random.default_rng(WINDOW_BENCHMARK_SEED + subject*1009 + repetition*100003)
    condition = ['clean','noisy','large_T','wide_QRS','irregular_RR'][subject % 5]
    fs = [250.0,500.0,1000.0][subject % 3]
    mean_rr = profile_rng.uniform(.50,1.10)
    amplitude = profile_rng.uniform(.40,1.60)
    width_scale = profile_rng.uniform(.8,1.4)
    if condition == 'wide_QRS':width_scale = profile_rng.uniform(1.8,2.6)
    t_ratio = profile_rng.uniform(.15,.45)
    if condition == 'large_T':t_ratio = profile_rng.uniform(.85,1.30)
    rotation, _ = np.linalg.qr(profile_rng.normal(size=(3,3)))
    times = np.arange(int(duration*fs))/fs
    xyz = np.zeros((len(times),3))
    references = []
    clock = 1.20
    while clock < duration-.8:
        references.append(clock)
        scale = amplitude*rng.uniform(.85,1.15)
        pulses = [(-.024*width_scale,.007*width_scale,[-.18,.05,.10]),
                  (0,.010*width_scale,[1.0,.60,.15]),
                  (.032*width_scale,.008*width_scale,[-.36,-.14,.18]),
                  (-min(.18,.28*mean_rr),.025,[.08,.045,-.025])]
        for center,sigma,vector in pulses:
            xyz += np.exp(-.5*((times-clock-center)/sigma)**2)[:,None]*scale*np.asarray(vector)
        t_center = min(.32,.40*mean_rr)
        t_sigma = min(.075,.13*mean_rr)
        if condition=='large_T':t_sigma = profile_rng.uniform(.028,.055)
        xyz += np.exp(-.5*((times-clock-t_center)/t_sigma)**2)[:,None]*scale*t_ratio*np.array([.8,.4,-.3])
        variation = .18 if condition=='irregular_RR' else .035
        clock += max(.35,mean_rr*(1+rng.normal(0,variation)))
    xyz = xyz @ rotation
    drift_amp = profile_rng.uniform(.02,.12)
    xyz += drift_amp*np.sin(2*np.pi*.25*times)[:,None]*np.array([1.,.5,-.3])
    noise_scale = profile_rng.uniform(.001,.005) if condition=='clean' else profile_rng.uniform(.008,.035)
    xyz += noise_scale*rng.normal(size=xyz.shape)
    xyz += profile_rng.uniform(.003,.025)*np.sin(2*np.pi*50*times)[:,None]*np.array([.6,1.,-.4])
    if condition=='noisy':
        burst = np.exp(-.5*((times-duration*.55)/.22)**2)
        xyz += .07*burst[:,None]*rng.normal(size=xyz.shape)
    return {'subject':subject,'repetition':repetition,'condition':condition,'fs':fs,
            'xyz':xyz,'reference_s':np.asarray(references),'duration_s':duration}


def synthetic_window_benchmark(windows=WINDOWS_MS, subjects_per_condition=9, repetitions=2):
    # Chia nhóm trước khi đánh giá; hai recording cùng subject chỉ thuộc một split.
    rng = np.random.default_rng(WINDOW_BENCHMARK_SEED)
    split = {}
    for condition_index in range(5):
        ids = np.arange(condition_index,subjects_per_condition*5,5)
        rng.shuffle(ids)
        n_validation = int(round(len(ids)*2/3))
        split.update({int(s):'validation' if j<n_validation else 'holdout' for j,s in enumerate(ids)})
    rows = []
    for subject in sorted(split):
        for repetition in range(repetitions):
            case = generate_xyz_case(subject,repetition)
            # Bỏ 1 giây mỗi biên cho cả nhãn lẫn prediction để tránh transient lọc.
            low, high = 1.0, case['duration_s']-1.0
            reference = case['reference_s'][(case['reference_s']>=low)&(case['reference_s']<=high)]
            morphology,detection = preprocess_xyz(case['xyz'],case['fs'],WINDOW_PREPROCESSING_CFG)
            for w in windows:
                error = ''
                try:
                    found = detect_qrs_anchors(morphology,detection,case['fs'],replace(WINDOW_PREPROCESSING_CFG,energy_window_ms=float(w)))
                    predicted = found['anchors']/case['fs']
                    predicted = predicted[(predicted>=low)&(predicted<=high)]
                except ValueError as exc:
                    predicted = np.array([]);error = str(exc)
                rows.append({'subject':subject,'repetition':repetition,'condition':case['condition'],
                             'fs':case['fs'],'split':split[subject],'window_ms':w,
                             'error':error,**match_qrs_events(predicted,reference,MATCH_TOLERANCE_MS)})
    details = pd.DataFrame(rows)
    summary = summarize_event_scores(details,['split','window_ms'])
    validation = summary[summary['split']=='validation'].sort_values(
        ['f1','matched_mae_ms','window_ms'],ascending=[False,True,True])
    selected = int(validation.iloc[0]['window_ms'])
    return details,summary,selected,split


def ptb_window_stability(data_root, windows=WINDOWS_MS):
    """Đếm nhịp/độ đồng thuận, không đánh giá accuracy vì PTB không có nhãn ở đây."""
    rows = [];errors = []
    paths = []
    for hea in sorted(Path(data_root).rglob('*.hea')):
        diagnosis = next((x.split(':',1)[1].strip().lower() for x in hea.read_text(encoding='latin-1').splitlines()
                          if x.lower().startswith('# reason for admission:')), '')
        if diagnosis in {'myocardial infarction','healthy control','healthy controls'}:
            paths.append((hea.with_suffix(''),'MI' if diagnosis=='myocardial infarction' else 'HC'))
    for number,(path,label) in enumerate(paths,1):
        rid = path.relative_to(data_root).as_posix()
        try:
            raw_case = load_xyz(path,CFG.seconds)
            morphology,detection = preprocess_xyz(raw_case['xyz'],raw_case['fs'],WINDOW_PREPROCESSING_CFG)
        except Exception as exc:
            errors.append({'record_id':rid,'error':str(exc)});continue
        found_by_window = {}
        for w in windows:
            try:
                found = detect_qrs_anchors(morphology,detection,raw_case['fs'],replace(WINDOW_PREPROCESSING_CFG,energy_window_ms=float(w)))
                found_by_window[w] = found
            except ValueError as exc:
                errors.append({'record_id':rid,'window_ms':w,'error':str(exc)})
        # Mode tính đối xứng từ tất cả cửa sổ; không đặt 80 ms làm ground truth.
        counts = [len(found_by_window[w]['anchors']) for w in found_by_window]
        modal = int(pd.Series(counts).mode().iloc[0]) if counts else np.nan
        for w,found in found_by_window.items():
            pair_f1 = []
            for other,other_found in found_by_window.items():
                if other==w:continue
                scores=match_qrs_events(found['anchors']/raw_case['fs'],other_found['anchors']/raw_case['fs'],30.0)
                denominator=2*scores['tp']+scores['fp']+scores['fn']
                pair_f1.append(2*scores['tp']/denominator if denominator else np.nan)
            rows.append({'record_id':rid,'patient_id':path.parent.name,'label':label,'window_ms':w,
                         'n_anchors':len(found['anchors']),'modal_count':modal,
                         'agrees_with_modal_count':len(found['anchors'])==modal,
                         'mean_pairwise_agreement':float(np.mean(pair_f1)),
                         'rr_warning':bool(found['flags']),'flags':'; '.join(found['flags'])})
        if number%100==0 or number==len(paths):print(f'PTB: {number}/{len(paths)} records')
    return pd.DataFrame(rows),pd.DataFrame(errors),len(paths)


def paired_subject_bootstrap(details, split, selected, baseline=80, repetitions=500):
    """CI của F1(selected)-F1(80), bootstrap theo subject, chỉ trên synthetic."""
    grouped = details[details['split']==split].groupby(['subject','window_ms'])[['tp','fp','fn']].sum()
    subjects = sorted(details.loc[details['split']==split,'subject'].unique())
    left=np.array([grouped.loc[(s,selected)].to_numpy() for s in subjects])
    right=np.array([grouped.loc[(s,baseline)].to_numpy() for s in subjects])
    rng=np.random.default_rng(WINDOW_BENCHMARK_SEED+771)
    deltas=[]
    def f1(a):return 2*a[0]/max(2*a[0]+a[1]+a[2],1)
    for _ in range(repetitions):
        sample=rng.integers(0,len(subjects),len(subjects))
        deltas.append(f1(left[sample].sum(axis=0))-f1(right[sample].sum(axis=0)))
    return np.quantile(deltas,[.025,.975])


### B. Nhìn trực tiếp đường bao khi đổi cửa sổ

Cùng một XYZ và cùng bộ lọc; chỉ độ rộng trung bình thay đổi. Các dấu × là mốc sau tinh chỉnh; đường đứt là ngưỡng thích nghi của từng cửa sổ. Đường bao thấp/cao hơn không tự cho biết detector tốt hơn.

Vùng phóng to được chọn ở nơi hai cửa sổ ngắn/dài bất đồng lớn nhất về mốc. Đây là lựa chọn để nhìn rõ sự khác nhau, không phải nhãn xác nhận mốc nào đúng. Số mốc in trong legend là số trong toàn recording đang phân tích.

In [ ]:
window_morph,window_detection=preprocess_xyz(raw_xyz,fs,WINDOW_PREPROCESSING_CFG)
visual_windows=[20,40,80,120,200]
visual_detections={w:detect_qrs_anchors(window_morph,window_detection,fs,replace(WINDOW_PREPROCESSING_CFG,energy_window_ms=w))
                   for w in visual_windows}
a_short=visual_detections[20]['anchors'];a_long=visual_detections[200]['anchors']
nearest=np.min(np.abs(a_short[:,None]-a_long[None,:]),axis=1)
zoom_center=float(a_short[int(np.argmax(nearest))]/fs)
zoom_left=max(0,zoom_center-.75);zoom_right=min(len(morph)/fs,zoom_center+.75)
timeline=np.arange(len(morph))/fs
mask=(timeline>=zoom_left)&(timeline<=zoom_right)
fig,axs=plt.subplots(len(visual_windows),1,figsize=(12,10),sharex=True,layout='constrained')
for ax,w in zip(axs,visual_windows):
    found=visual_detections[w]
    ax.plot(timeline[mask],found['envelope'][mask],color='#2563eb',label=f'{w} ms | toàn record: {len(found["anchors"])} mốc')
    ax.plot(timeline[mask],found['threshold'][mask],color='#dc2626',ls='--',lw=1,label='Ngưỡng')
    peaks=found['anchors'][(found['anchors']/fs>=zoom_left)&(found['anchors']/fs<=zoom_right)]
    ax.scatter(peaks/fs,found['envelope'][peaks],color='#ea580c',marker='x',s=55,label='Mốc phát hiện')
    visible_max=max(float(np.max(found['envelope'][mask])),float(np.max(found['threshold'][mask])),float(np.max(found['envelope'][peaks])) if len(peaks) else 0)
    ax.set_ylim(0,1.15*visible_max)
    ax.set_ylabel('mV');ax.legend(loc='upper left',fontsize=8,ncol=3);ax.grid(alpha=.2)
axs[-1].set_xlabel('Thời gian recording (s)')
fig.suptitle('Cùng dữ liệu, cửa sổ khác nhau → đường bao và mốc có thể khác nhau')
finish(fig)

### C. Chạy mọi cửa sổ trên tất cả MI/HC của PTB

`modal_count` là số mốc phổ biến nhất trong các cửa sổ thử của cùng record; nếu hòa lấy mode nhỏ nhất. Độ đồng thuận thời điểm dùng ghép một-một trong ±30 ms giữa từng cặp cửa sổ rồi lấy trung bình. Các đại lượng này được tính đối xứng từ mọi cửa sổ, không đặt 80 ms làm chuẩn.

Phép thử không đọc ECG II để tạo nhãn. Cờ RR chỉ là dấu hiệu cần xem lại, không xác nhận detector sai vì nhịp thật cũng có thể không đều. Thống kê PTB ở mức recording; một bệnh nhân có thể có nhiều recording.

In [ ]:
ptb_window_details,ptb_window_errors,n_ptb_attempted=ptb_window_stability(DATA_ROOT)
ptb_window_summary=ptb_window_details.groupby('window_ms').agg(
    n_records=('record_id','size'),total_anchors=('n_anchors','sum'),
    modal_count_agreement_pct=('agrees_with_modal_count',lambda a:100*a.mean()),
    pairwise_agreement_pct=('mean_pairwise_agreement',lambda a:100*a.mean()),
    rr_warning_records=('rr_warning','sum')).reset_index()
print('Record thử:',n_ptb_attempted,'| lỗi record/cửa sổ:',len(ptb_window_errors))
display(ptb_window_summary.round(3))
if not ptb_window_errors.empty:display(ptb_window_errors)

demo_counts=ptb_window_details[ptb_window_details['record_id'].isin([RECORD_ID,COMPARISON_RECORD])]
display(demo_counts.pivot(index='window_ms',columns='record_id',values='n_anchors'))
print('Các số ở đây là độ ổn định/đồng thuận, chưa phải accuracy.')

In [ ]:
fig,axs=plt.subplots(2,1,figsize=(12,7),sharex=True,layout='constrained')
for label,color in [('HC','#2563eb'),('MI','#ea580c')]:
    group=ptb_window_details[ptb_window_details['label']==label].groupby('window_ms')
    modal=group['agrees_with_modal_count'].mean()*100
    agreement=group['mean_pairwise_agreement'].mean()*100
    axs[0].plot(modal.index,modal.values,marker='o',color=color,label=label)
    axs[1].plot(agreement.index,agreement.values,marker='o',color=color,label=label)
for ax in axs:
    ax.axvline(80,color='.45',ls=':',label='80 ms đang dùng')
    ax.legend(fontsize=9);ax.grid(alpha=.2);ax.set_ylabel('%')
axs[0].set_title('PTB: tỷ lệ record có số mốc bằng mode của mọi cửa sổ')
axs[1].set_title('PTB: độ đồng thuận thời điểm giữa các cửa sổ — không phải accuracy')
axs[1].set_xlabel('Cửa sổ trung bình (ms)');axs[1].set_xticks(WINDOWS_MS)
finish(fig)

### D. Dữ liệu tổng hợp có mốc biết trước: đánh giá đúng, nhầm và bỏ sót

Các cửa sổ nhận cùng một bộ arrays và cùng reference. Generator dùng seed cố định, có tần số lấy mẫu 250/500/1000 Hz và các hình thái khác nhau. Các phép đo bên dưới mô tả đúng bộ tổng hợp này; không được chuyển thành tuyên bố độ chính xác trên PTB hoặc độ chính xác lâm sàng.

Validation dùng chọn cửa sổ; holdout chỉ dùng kiểm tra khả năng giữ kết quả trên subject tổng hợp khác. Toàn bộ grid được công bố, không chỉ công bố giá trị có lợi cho 80 ms.

In [ ]:
synthetic_window_details,synthetic_window_summary,selected_window_ms,synthetic_split=synthetic_window_benchmark()
display(synthetic_window_details[['subject','condition','fs','split']].drop_duplicates()
        .groupby(['split','condition']).size().rename('số subject').to_frame())
display(synthetic_window_summary[['split','window_ms','tp','fp','fn','precision','recall','f1','matched_mae_ms']].round(5))
print('Cửa sổ được chọn CHỈ trên validation tổng hợp:',selected_window_ms,'ms')
print('Tolerance ghép mốc:',MATCH_TOLERANCE_MS,'ms; MAE chỉ trên các mốc ghép được.')

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(13,4.5),sharey=True,layout='constrained')
for ax,split in zip(axs,['validation','holdout']):
    summary=synthetic_window_summary[synthetic_window_summary['split']==split]
    ax.plot(summary['window_ms'],100*summary['f1'],color='#2563eb',marker='o',label='F1 tổng hợp')
    eighty=summary[summary['window_ms']==80].iloc[0]
    ax.scatter(80,100*eighty['f1'],color='#ea580c',s=80,zorder=5,label=f'80 ms: {100*eighty["f1"]:.2f}%')
    ax.axvline(selected_window_ms,color='.4',ls=':',label='Chọn từ validation')
    ax.set(xlabel='Cửa sổ (ms)',title=split.upper()+' — dữ liệu tổng hợp')
    ax.grid(alpha=.2);ax.legend(fontsize=9);ax.set_xticks(WINDOWS_MS)
axs[0].set_ylabel('F1 (%)')
fig.suptitle('F1 trên dữ liệu tổng hợp: validation chọn tham số; holdout kiểm tra')
finish(fig)

fig,axs=plt.subplots(1,2,figsize=(13,4.5),layout='constrained')
for ax,split in zip(axs,['validation','holdout']):
    summary=synthetic_window_summary[synthetic_window_summary['split']==split]
    ax.plot(summary['window_ms'],summary['fp'],marker='o',color='#ea580c',label='FP: nhận nhầm')
    ax.plot(summary['window_ms'],summary['fn'],marker='s',color='#7c3aed',label='FN: bỏ sót')
    ax.axvline(80,color='.4',ls=':');ax.set(xlabel='Cửa sổ (ms)',ylabel='Số mốc',title=split.upper())
    ax.grid(alpha=.2);ax.legend(fontsize=9)
fig.suptitle('Đếm lỗi thực sự trên đồng hồ nhịp tổng hợp')
finish(fig)

### E. Kết quả có giữ được ở từng điều kiện và subject khác không

F1 tổng cao có thể che một kịch bản khó. Hình sau tách từng điều kiện tổng hợp trên holdout. Bảng bootstrap so F1(80 ms) − F1(cửa sổ khác) theo subject, có giữ hai recording cùng subject đi cùng nhau. Khoảng 95% chứa 0 nghĩa phép thử này chưa tách rõ hai lựa chọn; CI âm hoàn toàn nghiêng về cửa sổ khác.

Bootstrap này chỉ đo biến thiên của mô hình tổng hợp đã tạo, không lượng hóa bất định trên bệnh nhân thật. Không bootstrap 80 ms so với chính nó.

In [ ]:
scenario_summary=summarize_event_scores(
    synthetic_window_details[synthetic_window_details['split']=='holdout'],['condition','window_ms'])
fig,axes=plt.subplots(2,3,figsize=(13,7),sharex=True,sharey=True,layout='constrained')
condition_names={'clean':'Sạch','noisy':'Nhiễu','large_T':'T lớn','wide_QRS':'QRS rộng','irregular_RR':'RR không đều'}
for ax,(condition,group) in zip(axes.flat,scenario_summary.groupby('condition')):
    ax.plot(group['window_ms'],100*group['f1'],marker='o',color='#2563eb')
    ax.axvline(80,color='.4',ls=':')
    ax.set(xlabel='Cửa sổ (ms)',ylabel='F1 (%)',title=condition_names[condition])
    ax.set_xticks([20,80,120,200]);ax.grid(alpha=.2)
axes.flat[-1].set_visible(False)
fig.suptitle('Holdout tổng hợp theo điều kiện; đường dọc = 80 ms')
finish(fig)

ci_rows=[]
holdout=synthetic_window_summary[synthetic_window_summary['split']=='holdout'].set_index('window_ms')
for other in [60,100,200]:
    low,high=paired_subject_bootstrap(synthetic_window_details,'holdout',80,baseline=other)
    ci_rows.append({'so sánh':f'80 − {other} ms','ΔF1 (điểm %)':100*(holdout.loc[80,'f1']-holdout.loc[other,'f1']),
                    'CI 95% dưới (điểm %)':100*low,'CI 95% trên (điểm %)':100*high})
window_pairwise_ci=pd.DataFrame(ci_rows)
display(window_pairwise_ci.round(4))

### F. Kết luận tính trực tiếp từ kết quả, không chốt trước 80 ms

Ô dưới tự nêu cửa sổ tốt nhất theo tiêu chí đã đặt trên validation, và báo vị trí 80 ms trên holdout. Việc xem holdout không được dùng để thay lại tham số lựa chọn ở bước trước.

PTB không có nhãn chuẩn ở phép thử này, nên dù 80 ms có đồng thuận cao, chưa thể viết “80 ms tối ưu trên PTB”. Nếu 80 ms tốt nhất validation tổng hợp, cách viết đúng là **“80 ms được chọn trên bộ validation tổng hợp của phép thử này”**. Cần annotation độc lập trên dữ liệu thật để chốt lựa chọn cho nghiên cứu.

In [ ]:
validation_summary=synthetic_window_summary[synthetic_window_summary['split']=='validation'].set_index('window_ms')
holdout_summary=synthetic_window_summary[synthetic_window_summary['split']=='holdout'].set_index('window_ms')
best_holdout_f1=float(holdout_summary['f1'].max())
best_holdout_windows=holdout_summary.index[np.isclose(holdout_summary['f1'],best_holdout_f1,rtol=0,atol=1e-12)].tolist()
print(f'Validation tổng hợp chọn: {selected_window_ms} ms (F1={100*validation_summary.loc[selected_window_ms,"f1"]:.3f}%).')
print(f'80 ms trên holdout tổng hợp: F1={100*holdout_summary.loc[80,"f1"]:.3f}%.')
print(f'F1 cao nhất quan sát trên holdout: {best_holdout_windows} ms, {100*best_holdout_f1:.3f}%.')
if selected_window_ms==80:
    print('80 ms có bằng chứng hỗ trợ trong validation tổng hợp; chưa chứng minh tối ưu trên PTB thật.')
else:
    print('80 ms không đứng đầu validation tổng hợp của phép thử này; không có cơ sở gọi là tối ưu.')
if 80 not in best_holdout_windows:
    print('80 ms cũng không có F1 cao nhất trên holdout tổng hợp. Không kết luận tối ưu phổ quát.')
print('Giữ CFG=80 ms cho walkthrough hiện tại; không tự đổi pipeline theo kết quả holdout.')

experiment_dir=Path('vcg_window_validation');experiment_dir.mkdir(exist_ok=True)
ptb_window_details.to_csv(experiment_dir/'ptb_stability_details.csv',index=False)
ptb_window_summary.to_csv(experiment_dir/'ptb_stability_summary.csv',index=False)
synthetic_window_details.to_csv(experiment_dir/'synthetic_event_details.csv',index=False)
synthetic_window_summary.to_csv(experiment_dir/'synthetic_summary.csv',index=False)
window_pairwise_ci.to_csv(experiment_dir/'synthetic_paired_ci.csv',index=False)
print('Đã lưu dữ liệu số của các biểu đồ tại:',experiment_dir.resolve())

### Cơ sở phương pháp và giới hạn

[Pan & Tompkins, 1985 — paper gốc](https://courses.csail.mit.edu/18.337/2017/projects/subramanian_sandya/Papers/Pan%2BTompkins.pdf) mô tả tác động của độ rộng moving-window và chọn 150 ms thực nghiệm cho detector của họ. Detector notebook này có cấu trúc khác, nên không thừa hưởng tham số hay số liệu hiệu quả đó.

[Hướng dẫn WQRS của PhysioNet](https://physionet.org/physiotools/wag/wqrs-1.htm) minh họa đánh giá detector bằng cách so output với annotation tham chiếu. Ở đây chỉ dữ liệu tổng hợp có reference; phần PTB chỉ là kiểm tra độ ổn định.

Phép thử chỉ thay một tham số. Nó chưa tối ưu đồng thời band-pass, ngưỡng, refractory hoặc quy tắc tinh chỉnh. Bộ tổng hợp đơn giản có thể bỏ qua nhiều hình thái bệnh lý; do đó thứ hạng cửa sổ có thể thay đổi khi có annotation thật. Một cửa sổ tốt cho tìm nhịp cũng chưa bảo đảm biên QRS/T hoặc 12 feature đúng.